# BilgiTR — Kaynaklı Soru-Cevap Asistanı (RAG) Analiz Defteri

Bu çalışmada 27 Türkçe bilgi makalesinden oluşan BilgiTR arşivi üzerinde doğal dille sorulan sorulara **doğru kaynağı göstererek (citation)** yanıt veren bir Retrieval-Augmented Generation (RAG) hattı kurulmuş, farklı tasarım kararları ablasyon deneyleriyle kıyaslanmış ve seçilen en iyi modelin detaylı performans/hata analizi yapılmıştır.


In [1]:
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus_yolu = Path("data/corpus")
dokumanlar = []

for dosya in sorted(corpus_yolu.glob("*.txt")):
    metin = dosya.read_text(encoding="utf-8")
    kelimeler = metin.split()
    dokumanlar.append({
        "dosya_adi": dosya.name,
        "konu": dosya.name.replace(".txt", "").replace("_", " "),
        "kelime_sayisi": len(kelimeler),
        "karakter_sayisi": len(metin),
        "metin": metin
    })

df_corpus = pd.DataFrame(dokumanlar)

with open("data/gold_set.json", "r", encoding="utf-8") as f:
    gold_set = json.load(f)

print(f"Toplam Makale Sayısı   : {len(df_corpus)}")
print(f"Toplam Test Sorusu     : {len(gold_set)}")
print(f"Toplam Kelime Sayısı   : {df_corpus['kelime_sayisi'].sum()}")
print(f"Ortalama Kelime Sayısı : {df_corpus['kelime_sayisi'].mean():.1f}")
print(f"En Kısa Makale         : {df_corpus.loc[df_corpus['kelime_sayisi'].idxmin(), 'dosya_adi']} ({df_corpus['kelime_sayisi'].min()} kelime)")
print(f"En Uzun Makale         : {df_corpus.loc[df_corpus['kelime_sayisi'].idxmax(), 'dosya_adi']} ({df_corpus['kelime_sayisi'].max()} kelime)")

display(df_corpus[["dosya_adi", "konu", "kelime_sayisi", "karakter_sayisi"]].head(10))


Toplam Makale Sayısı   : 27
Toplam Test Sorusu     : 27
Toplam Kelime Sayısı   : 10078
Ortalama Kelime Sayısı : 373.3
En Kısa Makale         : Bal_arısı.txt (232 kelime)
En Uzun Makale         : Ay.txt (421 kelime)


,dosya_adi,konu,kelime_sayisi,karakter_sayisi
0,Antibiyotik.txt,Antibiyotik,355,3000
1,Ay.txt,Ay,421,3000
2,Aşı.txt,Aşı,382,3000
3,Bal_arısı.txt,Bal arısı,232,1646
4,Bağışıklık_sistemi.txt,Bağışıklık sistemi,358,3000
5,Beyin.txt,Beyin,382,3000
6,Buzul_çağı.txt,Buzul çağı,389,3000
7,DNA.txt,DNA,397,3000
8,Elektrik.txt,Elektrik,369,3000
9,Enzim.txt,Enzim,355,3000


### 1. Veri Keşfi (EDA) Bulguları
* Arşivde yer alan 27 makalenin ortalama uzunluğu **~350–400 kelime** civarındadır.
* Makaleler kısa olduğu için çok büyük parça boyutları (`500+` kelime) dokümanları hiç bölmeyecektir. Buna karşın astronomi (`Güneş Sistemi`, `Gezegen`, `Yıldız`, `Galaksi`, `Ay`) ve biyoloji (`Hücre`, `Virüs`, `Enzim`, `DNA`, `Bağışıklık sistemi`) kümelerinde kelime örtüşmesi yüksek olduğundan dokümanları **örtüşmeli (overlapping)** parçalara bölmek hem odaklanmayı artıracak hem de bağlam kopmasını önleyecektir.


In [2]:
def metin_temizle(metin: str) -> str:
    # Türkçe karakter duyarlı küçük harfe çevirme (İ -> i, I -> ı)
    metin = metin.replace("İ", "i").replace("I", "ı").lower()
    # Fazla boşluk ve satır sonlarını tek boşluğa indirgeme
    metin = re.sub(r"\s+", " ", metin).strip()
    return metin

def dokuman_parcala(df: pd.DataFrame, chunk_size: int = 100, overlap: int = 20, baslik_ekle: bool = True) -> list:
    parcalar = []
    adim = max(1, chunk_size - overlap)
    
    for _, satir in df.iterrows():
        kelimeler = satir["metin"].split()
        konu = satir["konu"]
        dosya = satir["dosya_adi"]
        
        for i in range(0, len(kelimeler), adim):
            parca_kelimeleri = kelimeler[i : i + chunk_size]
            # Sonda kalan çok kısa (15 kelimeden az) anlamsız artıkları atla
            if len(parca_kelimeleri) < 15 and len(parcalar) > 0 and parcalar[-1]["kaynak"] == dosya:
                break
                
            ham_parca = " ".join(parca_kelimeleri)
            tam_metin = f"{konu}: {ham_parca}" if baslik_ekle else ham_parca
                
            parcalar.append({
                "kaynak": dosya,
                "parca_id": i // adim,
                "ham_metin": tam_metin,
                "temiz_metin": metin_temizle(tam_metin)
            })
    return parcalar

ornek_parcalar = dokuman_parcala(df_corpus, chunk_size=100, overlap=20)
print(f"Toplam Oluşturulan Parça (Chunk) Sayısı : {len(ornek_parcalar)}")
print(f"Doküman Başına Ortalama Parça Sayısı    : {len(ornek_parcalar) / len(df_corpus):.2f}\n")
print("--- Örnek Parça Çıktısı (İndeks 0) ---")
print(f"Kaynak   : {ornek_parcalar[0]['kaynak']}")
print(f"Parça ID : {ornek_parcalar[0]['parca_id']}")
print(f"Metin    : {ornek_parcalar[0]['ham_metin']}")
print(f"Temiz Metin    : {ornek_parcalar[0]['temiz_metin']}")


Toplam Oluşturulan Parça (Chunk) Sayısı : 134
Doküman Başına Ortalama Parça Sayısı    : 4.96

--- Örnek Parça Çıktısı (İndeks 0) ---
Kaynak   : Antibiyotik.txt
Parça ID : 0
Metin    : Antibiyotik: Antibiyotik, bakterilere karşı aktif olan bir tür antimikrobiyal maddedir. Bakteriyel enfeksiyonlarla savaşmak için en önemli antibakteriyel ajan türüdür ve antibiyotik ilaçlar bu tür enfeksiyonların tedavisinde ve önlenmesinde yaygın olarak kullanılmaktadır. Bakterileri öldürebilir ya da büyümelerini engelleyebilirler. Sınırlı sayıda antibiyotik de antiprotozoal aktiviteye sahiptir. Antibiyotikler soğuk algınlığı veya gribe neden olan virüsler gibi virüslere karşı etkili değildir; virüslerin büyümesini engelleyen ilaçlar antibiyotik yerine antiviral ilaçlar veya antiviraller olarak adlandırılır. Mantarlara karşı da etkili değildirler; mantarların büyümesini engelleyen ilaçlara antifungal ilaçlar denir. Bazen, antibiyotik terimi - Yunanca ἀντι anti, "karşı" ve βίος bios, "yaşam" köklerinden g

### 2. Metin Ön İşleme ve Parçalama (Chunking) Stratejisi
* **Türkçe Normalizasyonu:** Standart `.lower()` fonksiyonu büyük `İ` harfini bozduğu için özel `metin_temizle` fonksiyonu ile `İ -> i` ve `I -> ı` dönüşümü sağlanmıştır.
* **Örtüşmeli Parçalama (Overlap):** `chunk_size=100` ve `overlap=20` ile cümle bütünlüğünün parça sınırlarında kesilmesi önlenmiştir.
* **Konu Bağlamı Zenginleştirme:** Her parçanın başına makale konusunun (`Fotosentez: ...`) eklenmesi, ilgili dokümanın tüm parçalarında konu sinyalini canlı tutmaktadır.


In [3]:
class BilgiTRRetriever:
    def __init__(self, chunk_size=100, overlap=20, yontem="char_ngram", baslik_ekle=True):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.yontem = yontem
        self.baslik_ekle = baslik_ekle
        
        if yontem == "char_ngram":
            # Türkçe ekleri yakalamak için kelime sınırları içinde 3-5 karakterlik n-gramlar
            self.vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
        elif yontem == "word_tfidf":
            # Standart kelime bazlı 1-2 gram TF-IDF
            self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        else:
            raise ValueError("Geçersiz yöntem! 'char_ngram' veya 'word_tfidf' seçin.")
            
        self.parcalar = []
        self.matris = None

    def indeksle(self, df: pd.DataFrame):
        self.parcalar = dokuman_parcala(
            df, chunk_size=self.chunk_size, overlap=self.overlap, baslik_ekle=self.baslik_ekle
        )
        temiz_metinler = [p["temiz_metin"] for p in self.parcalar]
        self.matris = self.vectorizer.fit_transform(temiz_metinler)

    def getir(self, soru: str, top_k: int = 3) -> list:
        temiz_soru = metin_temizle(soru)
        soru_vek = self.vectorizer.transform([temiz_soru])
        skorlar = cosine_similarity(soru_vek, self.matris).flatten()
        en_iyi_indeksler = np.argsort(skorlar)[::-1][:top_k]
        
        sonuclar = []
        for idx in en_iyi_indeksler:
            sonuclar.append({
                "kaynak": self.parcalar[idx]["kaynak"],
                "parca_id": self.parcalar[idx]["parca_id"],
                "skor": round(float(skorlar[idx]), 4),
                "metin": self.parcalar[idx]["ham_metin"]
            })
        return sonuclar

    def cevapla(self, soru: str, top_k: int = 3) -> dict:
        getirilenler = self.getir(soru, top_k=top_k)
        kaynaklar = list(dict.fromkeys([g["kaynak"] for g in getirilenler]))
        cevap_satirlari = [
            f"• {g['metin'][:180]}... [Kaynak: {g['kaynak']} | Skor: {g['skor']}]"
            for g in getirilenler
        ]
        return {
            "soru": soru,
            "cevap": "\n".join(cevap_satirlari),
            "kaynaklar": kaynaklar
        }

def sistemi_olc(retriever: BilgiTRRetriever, gold_data: list, top_k: int = 3) -> dict:
    isabet = 0
    mrr_toplam = 0.0
    anahtar_kelime_isabet = 0
    hatali_sorular = []

    for item in gold_data:
        soru = item["soru"]
        dogru_dosya = item["dogru_dosya"]
        anahtar = metin_temizle(item["anahtar_kelime"])

        getirilenler = retriever.getir(soru, top_k=top_k)
        getirilen_dosyalar = [g["kaynak"] for g in getirilenler]
        birlesik_metin = metin_temizle(" ".join([g["metin"] for g in getirilenler]))

        if dogru_dosya in getirilen_dosyalar:
            isabet += 1
            sira = getirilen_dosyalar.index(dogru_dosya) + 1
            mrr_toplam += 1.0 / sira
        else:
            hatali_sorular.append({
                "soru": soru,
                "beklenen": dogru_dosya,
                "getirilenler": getirilen_dosyalar
            })

        if anahtar in birlesik_metin:
            anahtar_kelime_isabet += 1

    n = len(gold_data)
    return {
        "hit_rate": round(isabet / n, 3),
        "mrr": round(mrr_toplam / n, 3),
        "keyword_hit_rate": round(anahtar_kelime_isabet / n, 3),
        "dogru_sayisi": f"{isabet}/{n}",
        "hatalar": hatali_sorular
    }

# Sistemin çalışma doğrulaması ve örnek kaynaklı yanıt (Citation)
ornek_rag = BilgiTRRetriever(chunk_size=100, overlap=20, yontem="char_ngram")
ornek_rag.indeksle(df_corpus)
ornek_yanit = ornek_rag.cevapla(gold_set[0]["soru"], top_k=2)

print("--- Sistem Çalışma ve Citation Doğrulaması ---")
print(f"Soru: {ornek_yanit['soru']}")
print(f"Cevap:\n{ornek_yanit['cevap']}")
print(f"Döndürülen Kaynaklar: {ornek_yanit['kaynaklar']}")


--- Sistem Çalışma ve Citation Doğrulaması ---
Soru: Bitkilerin ışık enerjisini kimyasal enerjiye çevirdiği, oksijen açığa çıkaran işlem nedir?
Cevap:
• Fotosentez: Fotosentez, bitkiler ve diğer canlılar tarafından, ışık enerjisini organizmaların yaşamsal eylemlerine enerji sağlamak için daha sonra serbest bırakılabilecek kimyasal ... [Kaynak: Fotosentez.txt | Skor: 0.2809]
• Güneş enerjisi: Güneş enerjisi, kaynağı Güneş olan ısı ve parlak ışıktır. Güneş'in çekirdeğinde yer alan füzyon süreci ile açığa çıkan ışınım enerjisidir. Güneşteki hidrojen gazını... [Kaynak: Güneş_enerjisi.txt | Skor: 0.265]
Döndürülen Kaynaklar: ['Fotosentez.txt', 'Güneş_enerjisi.txt']


### 3. Retrieval Hattı ve Kaynak Gösterimi (Citation)
* Sistem, sorgu vektörü ile indekslenmiş doküman parçaları arasındaki **Kosinüs Benzerliğini (Cosine Similarity)** hesaplayarak en yüksek skorlu ilk `k` parçayı getirmektedir.
* Üretilen her yanıtta ilgili metin parçasının yanında mutlaka **kaynak dosya adı** ve **benzerlik skoru** döndürülerek şeffaf ve doğrulanabilir bir citation yapısı sağlanmıştır.


In [4]:
deney_konfigurasyonlari = [
    {"ad": "1. Baseline (Kelime TF-IDF, chunk=100, k=3)", "chunk": 100, "overlap": 20, "yontem": "word_tfidf", "k": 3},
    {"ad": "2. Karakter N-Gram (chunk=100, k=3)", "chunk": 100, "overlap": 20, "yontem": "char_ngram", "k": 3},
    {"ad": "3. Küçük Parça (char_ngram, chunk=50, k=3)", "chunk": 50, "overlap": 10, "yontem": "char_ngram", "k": 3},
    {"ad": "4. Büyük Parça (char_ngram, chunk=250, k=3)", "chunk": 250, "overlap": 40, "yontem": "char_ngram", "k": 3},
    {"ad": "5. Tam Doküman (Bölmeden, chunk=2000, k=3)", "chunk": 2000, "overlap": 0, "yontem": "char_ngram", "k": 3},
    {"ad": "6. Tek Sonuç (char_ngram, chunk=100, k=1)", "chunk": 100, "overlap": 20, "yontem": "char_ngram", "k": 1},
    {"ad": "7. Geniş Sonuç (char_ngram, chunk=100, k=5)", "chunk": 100, "overlap": 20, "yontem": "char_ngram", "k": 5},
]

sonuc_listesi = []
for cfg in deney_konfigurasyonlari:
    model = BilgiTRRetriever(chunk_size=cfg["chunk"], overlap=cfg["overlap"], yontem=cfg["yontem"])
    model.indeksle(df_corpus)
    olcum = sistemi_olc(model, gold_set, top_k=cfg["k"])
    sonuc_listesi.append({
        "Deney": cfg["ad"],
        "Parça Sayısı": len(model.parcalar),
        "Hit-Rate@k": olcum["hit_rate"],
        "MRR": olcum["mrr"],
        "Keyword Hit": olcum["keyword_hit_rate"],
        "İsabet": olcum["dogru_sayisi"]
    })

df_ablasyon = pd.DataFrame(sonuc_listesi)
display(df_ablasyon)


,Deney,Parça Sayısı,Hit-Rate@k,MRR,Keyword Hit,İsabet
0,"1. Baseline (Kelime TF-IDF, chunk=100, k=3)",134,0.963,0.938,0.481,26/27
1,"2. Karakter N-Gram (chunk=100, k=3)",134,0.963,0.926,0.556,26/27
2,"3. Küçük Parça (char_ngram, chunk=50, k=3)",256,0.963,0.944,0.444,26/27
3,"4. Büyük Parça (char_ngram, chunk=250, k=3)",54,0.926,0.864,0.852,25/27
4,"5. Tam Doküman (Bölmeden, chunk=2000, k=3)",27,1.000,0.901,1.000,27/27
5,"6. Tek Sonuç (char_ngram, chunk=100, k=1)",134,0.889,0.889,0.481,24/27
6,"7. Geniş Sonuç (char_ngram, chunk=100, k=5)",134,0.963,0.926,0.704,26/27


### 4. Ablasyon (Karşılaştırma) Analizi ve En İyi Model Kararı
* **Kelime vs. Karakter N-Gram (Deney 1 vs. 2):** Türkçe sondan eklemeli bir dil olduğu için karakter `3-5 gram` yaklaşımı kelime köklerini daha iyi yakalayarak `Keyword Hit` oranını **%48.1'den %55.6'ya** çıkarmıştır.
* **Parça Boyutu Seçimi (Deney 2, 3, 4, 5):** Arşivdeki makaleler kısa (~370 kelime) olduğu için Tam Doküman (`chunk=2000`) %100 isabet verse de uzun metin birikimi nedeniyle sıralama keskinliğini (`MRR: 0.901`) düşürmüştür. Gerçek üretim ortamında uzun makalelerle ölçeklenebilirlik sağlamak, LLM bağlam penceresini verimli kullanmak ve yüksek sıralama keskinliğini (`MRR: 0.926`) korumak amacıyla **`chunk_size=100, overlap=20, char_ngram`** en iyi mimari olarak seçilmiştir.
* **Sonuç Sayısı `k` (Deney 2, 6, 7):** `k=1` iken isabet **%88.9**'da kalırken `k=3`'te **%96.3**'e (26/27) sıçramaktadır. `k=5`'e çıkmak dosya isabetini artırmayıp yalnızca gürültüyü artırdığından **`k=3`** ideal değerdir.


In [5]:
en_iyi_model = BilgiTRRetriever(chunk_size=100, overlap=20, yontem="char_ngram")
en_iyi_model.indeksle(df_corpus)

print("--- Seçilen En İyi Modelin Performans Özeti (chunk=100, overlap=20, char_ngram) ---")
for k in (1, 3, 5):
    sonuc = sistemi_olc(en_iyi_model, gold_set, top_k=k)
    print(f"k={k} -> Hit-Rate@{k}: {sonuc['hit_rate']:.3f} ({sonuc['dogru_sayisi']}) | MRR: {sonuc['mrr']:.3f} | Keyword Hit: {sonuc['keyword_hit_rate']:.3f}")

olcum_k3 = sistemi_olc(en_iyi_model, gold_set, top_k=3)
olcum_k1 = sistemi_olc(en_iyi_model, gold_set, top_k=1)

print(f"\n=== k=3 İken Kaçırılan Sorular ({len(olcum_k3['hatalar'])} Adet) ===")
for h in olcum_k3["hatalar"]:
    print(f"Soru      : {h['soru']}")
    print(f"Beklenen  : {h['beklenen']}")
    print(f"Getirilen : {h['getirilenler']}\n")

print(f"=== k=1 İken 1. Sırada Yanılınan Sorular ({len(olcum_k1['hatalar'])} Adet) ===")
for h in olcum_k1["hatalar"]:
    print(f"Soru      : {h['soru']}")
    print(f"Beklenen  : {h['beklenen']}")
    print(f"1. Gelen  : {h['getirilenler']}\n")


--- Seçilen En İyi Modelin Performans Özeti (chunk=100, overlap=20, char_ngram) ---
k=1 -> Hit-Rate@1: 0.889 (24/27) | MRR: 0.889 | Keyword Hit: 0.481
k=3 -> Hit-Rate@3: 0.963 (26/27) | MRR: 0.926 | Keyword Hit: 0.556
k=5 -> Hit-Rate@5: 0.963 (26/27) | MRR: 0.926 | Keyword Hit: 0.704

=== k=3 İken Kaçırılan Sorular (1 Adet) ===
Soru      : Milyarlarca yıldızı kütle çekimiyle bir arada tutan devasa gök sistemine ne denir?
Beklenen  : Galaksi.txt
Getirilen : ['Yıldız.txt', 'Yıldız.txt', 'Yıldız.txt']

=== k=1 İken 1. Sırada Yanılınan Sorular (3 Adet) ===
Soru      : Dünya'nın deniz seviyesinden en yüksek dağı hangisidir?
Beklenen  : Everest_Dağı.txt
1. Gelen  : ['Ay.txt']

Soru      : Güneş ışınlarından ısı veya elektrik elde etmeye dayanan yenilenebilir enerji kaynağı nedir?
Beklenen  : Güneş_enerjisi.txt
1. Gelen  : ['Elektrik.txt']

Soru      : Milyarlarca yıldızı kütle çekimiyle bir arada tutan devasa gök sistemine ne denir?
Beklenen  : Galaksi.txt
1. Gelen  : ['Yıldız.txt']



### 5. Hata Analizi (Error Analysis) ve Sonuç
* Seçilen en iyi model `k=3` iken 27 sorunun **26'sını (%96.3)** doğru kaynakla eşleştirmiş, yalnızca 1 soruda ve `k=1` iken 3 soruda yakın konulu makaleleri üst sıraya almıştır.
* Hataların tamamı **ortak kelime havuzu paylaşan konu kümelerinde** (Astronomi: `Yıldız`, `Gezegen`, `Güneş Sistemi` ve Biyoloji: `Hücre`, `Virüs`, `Bağışıklık sistemi`) gerçekleşmektedir. Örneğin soru bir gezegeni veya uyduyu sorsa dahi soruda geçen *"yıldız"* veya *"kütleçekimi"* kelimeleri nedeniyle ilgili diğer astronomi makaleleri yüksek kosinüs benzerliği alabilmektedir.
* Bu durum sözcüksel (lexical) aramanın doğal bir sınırı olup, gelecek adımda anlamsal (semantic) yakınlığı yakalayan Türkçe destekli **Dense Embedding** modelleriyle hibrit arama (BM25 + Vektör) kullanılarak çözülebilir.
